# Clase 11 — Búsqueda y recuperación clásica

**Curso:** NLP Clásico con Python · **Bloque 4:** Significado y búsqueda
**Tiempo estimado:** 105 minutos · **Capítulo del manual:** `manual/11_busqueda_recuperacion_clasica.md`

## Qué vas a ver en esta clase

- La **recuperación de información** como problema de ordenar documentos por relevancia frente a una consulta.
- Un motor de búsqueda con **TF-IDF y similitud del coseno**, programado con NumPy, sobre un ejemplo pequeño calculado a mano.
- **BM25** (`k1`, `b`), también programado con NumPy: por qué su IDF probabilístico puede ser **negativo** cuando un término aparece en más de la mitad de los documentos, y cómo se corrige.
- Evaluación con **precisión@k** y **MRR** sobre una colección de preguntas frecuentes (FAQ) con relevancia fijada de antemano.
- Un caso concreto en el que la búsqueda por palabras **falla** ante sinónimos y paráfrasis: la motivación para los embeddings densos como recuperador alternativo.

**Requisitos:** clase 4 (TF-IDF, bolsa de palabras); la intuición del coseno de la clase 10 ayuda pero no es imprescindible. **Librerías:** `numpy`, `pandas`, `nltk` (solo para las listas de stopwords). **Datos:** `datos/faq_es.csv`, `datos/faq_it.csv` (colección de FAQ) y `datos/faq_consultas_es.csv`, `datos/faq_consultas_it.csv` (consultas con relevancia fijada de antemano); en Colab, sube la carpeta `datos/`. Todas las cifras son un *ejemplo ilustrativo*, calculadas sobre una colección sintética de 40 documentos por idioma. **Duración de la ejecución:** unos segundos.

---
## Celda 0 — Setup

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Instala el paquete solo si falta (idempotente y silencioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("matplotlib", "matplotlib")
print("Librerías listas.")

In [ ]:
import os
import re
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
np.random.seed(42)                        # semilla fija (aunque este tema no entrena nada aleatorio)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True

try:
    import nltk
    from nltk.corpus import stopwords
    stopwords.words("spanish")
except LookupError:
    nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
STOPWORDS = {"es": set(stopwords.words("spanish")), "it": set(stopwords.words("italian"))}   # dato-es

def cargar_datos(nombre):
    """Busca un archivo de datos en datos/, ../datos/ y /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "No encuentro " + nombre + ". Sube la carpeta datos/ del curso (en Colab: panel de archivos) o déjala junto al notebook.")

def tokenizar(texto, idioma, quitar_stopwords=True):
    """Minúsculas, solo letras (sin números ni puntuación), y stopwords opcionales."""
    palabras = re.findall(r"[^\W\d_]+", texto.lower())
    if quitar_stopwords:
        palabras = [p for p in palabras if p not in STOPWORDS[idioma]]
    return palabras

def construir_vocabulario(documentos_tok):
    """Lista ordenada de palabras distintas que aparecen en al menos un documento."""
    vocab = sorted({p for doc in documentos_tok for p in doc})
    return vocab, {p: i for i, p in enumerate(vocab)}

def matriz_tf(documentos_tok, indice):
    """Matriz de recuentos (documentos x vocabulario): TF crudo, sin normalizar."""
    M = np.zeros((len(documentos_tok), len(indice)), dtype=float)
    for i, doc in enumerate(documentos_tok):
        for palabra, n in Counter(doc).items():
            if palabra in indice:
                M[i, indice[palabra]] = n
    return M

def idf_tfidf(TF):
    """IDF clásico y suavizado: log((1+N)/(1+df)) + 1, siempre >= 1 (nunca negativo)."""
    N = TF.shape[0]
    df = (TF > 0).sum(axis=0)
    return np.log((1 + N) / (1 + df)) + 1.0

def matriz_tfidf(TF, idf):
    """TF-IDF = TF * IDF, con cada fila normalizada a norma 1 (para que el coseno sea un producto escalar)."""
    X = TF * idf
    normas = np.linalg.norm(X, axis=1, keepdims=True)
    normas[normas == 0] = 1.0
    return X / normas

def vector_consulta(consulta_tok, indice, idf):
    """Mismo vocabulario e IDF que los documentos; TF de la consulta = recuento de cada palabra."""
    v = np.zeros(len(indice))
    for palabra, n in Counter(consulta_tok).items():
        if palabra in indice:
            v[indice[palabra]] = n
    v = v * idf
    norma = np.linalg.norm(v)
    return v / norma if norma > 0 else v

def buscar_coseno(consulta_tok, X, indice, idf, k=5):
    """Ordena los documentos por coseno con la consulta; devuelve (índices, puntuaciones)."""
    q = vector_consulta(consulta_tok, indice, idf)
    puntuaciones = X @ q
    orden = np.argsort(-puntuaciones)[:k]
    return orden, puntuaciones[orden]

def idf_bm25(TF, corregido=False):
    """IDF probabilístico de Robertson-Sparck Jones: log((N-df+0.5)/(df+0.5)); puede ser NEGATIVO si df > N/2. Con corregido=True, log(1 + (N-df+0.5)/(df+0.5)), siempre >= 0."""
    N = TF.shape[0]
    df = (TF > 0).sum(axis=0)
    razon = (N - df + 0.5) / (df + 0.5)
    return np.log(1.0 + razon) if corregido else np.log(razon)

def bm25_matriz(TF, idf, k1=1.5, b=0.75):
    """Matriz BM25 (documentos x vocabulario): para cada palabra y documento, el término que se suma a la puntuación si la palabra está en la consulta."""
    longitudes = TF.sum(axis=1, keepdims=True)
    long_media = longitudes.mean()
    denom = TF + k1 * (1 - b + b * longitudes / long_media)
    return idf * (TF * (k1 + 1) / np.where(denom == 0, 1, denom))

def buscar_bm25(consulta_tok, BM25, indice, k=5):
    """Suma, para cada documento, la columna BM25 de cada palabra de la consulta que está en el vocabulario."""
    cols = [indice[p] for p in consulta_tok if p in indice]
    puntuaciones = BM25[:, cols].sum(axis=1) if cols else np.zeros(BM25.shape[0])
    orden = np.argsort(-puntuaciones)[:k]
    return orden, puntuaciones[orden]

def precision_en_k(recuperados, relevantes, k):
    """Fracción de relevantes entre los k primeros recuperados."""
    top = recuperados[:k]
    return len([d for d in top if d in relevantes]) / k

def rango_reciproco(recuperados, relevantes):
    """1 / posición (1-indexada) del primer documento relevante; 0 si no aparece."""
    for i, d in enumerate(recuperados, 1):
        if d in relevantes:
            return 1.0 / i
    return 0.0

def palabras_clave(texto, idioma, TF_global, idf_global, indice_global, n=5):
    """Las n palabras del documento con mayor TF-IDF (TF local del documento, IDF de todo el corpus)."""
    tok = tokenizar(texto, idioma)
    cuenta = Counter(tok)
    puntuadas = [(p, n_ * idf_global[indice_global[p]]) for p, n_ in cuenta.items() if p in indice_global]
    puntuadas.sort(key=lambda par: -par[1])
    return [p for p, _ in puntuadas[:n]]

def dividir_frases(texto):
    """Corta un texto en frases por . ! ? y descarta las vacías."""
    return [f.strip() for f in re.split(r"(?<=[.!?])\s+", texto.strip()) if f.strip()]

def resumen_extractivo(texto, idioma, idf_global, indice_global, n_frases=2):
    """Puntúa cada frase con la media del TF-IDF de sus palabras (TF dentro de la frase) y devuelve las n_frases mejores, en su orden original."""
    frases = dividir_frases(texto)
    puntuaciones = []
    for frase in frases:
        tok = tokenizar(frase, idioma)
        if not tok:
            puntuaciones.append(0.0)
            continue
        cuenta = Counter(tok)
        pesos = [cuenta[p] * idf_global[indice_global[p]] for p in cuenta if p in indice_global]
        puntuaciones.append(float(np.mean(pesos)) if pesos else 0.0)
    mejores = sorted(range(len(frases)), key=lambda i: -puntuaciones[i])[:n_frases]
    mejores.sort()
    return [frases[i] for i in mejores], puntuaciones

print("Funciones auxiliares cargadas.")

---
## 1. TF-IDF y coseno: motor de búsqueda a mano

Cuatro documentos diminutos (*ejemplo ilustrativo*) para calcular TF-IDF a mano y comprobarlo con código.

In [ ]:
# datos-es
# Documento diminuto para calcular TF-IDF y BM25 a mano (idéntico en ES e IT: son datos)
TOY = ["el gato come pescado", "el gato come pescado pescado pescado", "el perro corre mucho", "el pescado nada"]

In [ ]:
toy_tok = [t.split() for t in TOY]                       # tokenización trivial: ya están en minúsculas y sin puntuación
vocab_t, idx_t = construir_vocabulario(toy_tok)
TFt = matriz_tf(toy_tok, idx_t)
idf_t = idf_tfidf(TFt)
print("vocabulario:", vocab_t)
print("df:", {w: int((TFt[:, idx_t[w]] > 0).sum()) for w in vocab_t})
print("idf(pescado) =", round(float(idf_t[idx_t["pescado"]]), 3), "| idf(gato) =", round(float(idf_t[idx_t["gato"]]), 3))
assert vocab_t.count("pescado") == 1 and int((TFt[:, idx_t["pescado"]] > 0).sum()) == 3
assert np.isclose(idf_t[idx_t["pescado"]], np.log(5 / 4) + 1)          # cálculo a mano

### ¿Qué observas?

`pescado` aparece en 3 de los 4 documentos: `idf = log(5/4) + 1 ≈ 1,223`, tal como se calculó a mano en el capítulo. El IDF de esta fórmula (la que usa `TfidfVectorizer` por defecto) **nunca es negativo**: como mucho vale 1 cuando la palabra está en todos los documentos.

In [ ]:
X_t = matriz_tfidf(TFt, idf_t)
q_tok = tokenizar("pescado", "es", quitar_stopwords=False)   # dato-es
orden, punt = buscar_coseno(q_tok, X_t, idx_t, idf_t, k=4)
for i, p in zip(orden, punt):
    print(f"  {TOY[i]!r:45} -> {p:.3f}")
assert list(orden) == [1, 3, 0, 2]     # el doc que más repite "pescado" sale primero; el único sin esa palabra queda último

---
## 2. BM25: la fórmula, y el problema del IDF negativo

Con los mismos 4 documentos, el IDF probabilístico de BM25 (`log((N-df+0,5)/(df+0,5))`) para `pescado` (`df=3` de `N=4`, más de la mitad) y para `el` (`df=4`, todos los documentos):

In [ ]:
idf_b = idf_bm25(TFt)
idf_b_corr = idf_bm25(TFt, corregido=True)
palabras_toy = ("pescado", "el", "corre")   # dato-es
for w in palabras_toy:
    print(f"  {w:>8} df={int((TFt[:, idx_t[w]] > 0).sum())}  idf_bm25={idf_b[idx_t[w]]:.3f}  idf_bm25_corregido={idf_b_corr[idx_t[w]]:.3f}")
palabra_el = "el"   # dato-es
assert idf_b[idx_t["pescado"]] < 0 and idf_b[idx_t[palabra_el]] < 0        # df > N/2 -> idf negativo
assert idf_b_corr[idx_t["pescado"]] > 0 and idf_b_corr[idx_t[palabra_el]] > 0

### ¿Qué observas?

`el` está en los 4 documentos: `idf_bm25 = log(0,5/4,5) = -2,197`. `pescado` está en 3 de 4: `idf_bm25 = log(1,5/3,5) = -0,847`. Ambos negativos, tal como se explica en el capítulo (sección 3.1): en cuanto `df > N/2`, el logaritmo de un número menor que 1 es negativo. `corre` (`df=1`) da un `idf_bm25` positivo. Con el IDF corregido (`log(1+...)`), los tres son positivos.

**El problema en la práctica:** con `idf` negativo, la puntuación BM25 del término `pescado` en cada documento queda invertida.

In [ ]:
k1, b = 1.5, 0.75
BM25_malo = bm25_matriz(TFt, idf_b, k1=k1, b=b)
BM25_bueno = bm25_matriz(TFt, idf_b_corr, k1=k1, b=b)
i_p = idx_t["pescado"]
print("longitudes de los documentos:", TFt.sum(axis=1).astype(int), "| media:", round(float(TFt.sum(axis=1).mean()), 2))
print("BM25(pescado) con idf SIN corregir:", np.round(BM25_malo[:, i_p], 3))
print("BM25(pescado) con idf corregido:   ", np.round(BM25_bueno[:, i_p], 3))
orden_malo = np.argsort(-BM25_malo[:, i_p])
orden_bueno = np.argsort(-BM25_bueno[:, i_p])
print("orden SIN corregir, del peor al mejor:", TOY[orden_malo[-1]], "...", TOY[orden_malo[0]], "(primero)")
print("orden corregido, primero:", TOY[orden_bueno[0]])
assert orden_malo[0] == 2 and BM25_malo[2, i_p] == 0.0     # el doc SIN "pescado" queda primero: está mal
assert orden_bueno[0] == 1                                  # con el idf corregido, el doc que más repite "pescado" queda primero
assert BM25_malo[1, i_p] < BM25_malo[0, i_p] < BM25_malo[2, i_p]   # orden invertido con idf sin corregir

### ¿Qué observas?

Con el IDF sin corregir, el documento que **no** contiene `pescado` (`el perro corre mucho`) obtiene puntuación 0 y queda **primero**, por delante de los tres documentos que sí lo mencionan; el que más lo repite queda último. Con el IDF corregido, el orden se arregla: el documento que más repite `pescado` (ajustado por su longitud) queda primero. Es exactamente la tabla de la sección 3 del capítulo.

---
## 3. ¿Y en un corpus real? Stopwords y el IDF negativo

Cargamos la colección de FAQ (40 documentos por idioma) y comprobamos cuántas palabras tienen `idf_bm25` negativo, **con y sin quitar las stopwords** al tokenizar.

In [ ]:
IDIOMAS = ("es", "it")   # dato-es
FAQ, TOK, VOCAB, IDX_C, TF_C, IDF_TFIDF, X_C, IDF_BM25, BM25_C = {}, {}, {}, {}, {}, {}, {}, {}, {}
for idioma in IDIOMAS:
    FAQ[idioma] = pd.read_csv(cargar_datos(f"faq_{idioma}.csv"))
    docs = (FAQ[idioma]["titulo"] + ". " + FAQ[idioma]["texto"]).tolist()
    tok_sin = [tokenizar(d, idioma, quitar_stopwords=False) for d in docs]
    vocab_sin, idx_sin = construir_vocabulario(tok_sin)
    TF_sin = matriz_tf(tok_sin, idx_sin)
    idf_sin = idf_bm25(TF_sin)
    negativas_sin = int((idf_sin < 0).sum())
    TOK[idioma] = [tokenizar(d, idioma) for d in docs]                 # con stopwords quitadas: el pipeline real del capítulo
    VOCAB[idioma], IDX_C[idioma] = construir_vocabulario(TOK[idioma])
    TF_C[idioma] = matriz_tf(TOK[idioma], IDX_C[idioma])
    IDF_TFIDF[idioma] = idf_tfidf(TF_C[idioma])
    X_C[idioma] = matriz_tfidf(TF_C[idioma], IDF_TFIDF[idioma])
    IDF_BM25[idioma] = idf_bm25(TF_C[idioma])
    BM25_C[idioma] = bm25_matriz(TF_C[idioma], IDF_BM25[idioma])
    negativas_con = int((IDF_BM25[idioma] < 0).sum())
    print(idioma, "| docs:", len(FAQ[idioma]), "| vocabulario:", len(VOCAB[idioma]),
          "| palabras con idf_bm25 < 0 SIN quitar stopwords:", negativas_sin, "de", len(vocab_sin),
          "| CON stopwords quitadas:", negativas_con, "de", len(VOCAB[idioma]))
    assert negativas_sin > 0 and negativas_con == 0     # el resultado del capítulo, sección 4

### ¿Qué observas?

Sin quitar stopwords, varias palabras superan `df > N/2 = 20` (`de`, `y`, `la`… en español; `e`, `di`, `in`… en italiano) y su `idf_bm25` es negativo. Quitando las stopwords antes de construir el vocabulario, ninguna de las 556 (español) o 584 (italiano) palabras restantes llega a superar la mitad de los documentos: el problema desaparece en esta colección, aunque no hay garantía de que eso ocurra siempre (por eso el código de este capítulo usa el IDF corregido de todos modos, sección 2).

---
## 4. Evaluación: precisión@k y MRR

Las consultas y su documento relevante están fijados de antemano en `datos/faq_consultas_es.csv` / `_it.csv` (columna `tipo`: `facil` o `dificil`). Buscamos con TF-IDF+coseno y con BM25 (IDF corregido) y calculamos precisión@3 y MRR, separando por tipo de consulta.

In [ ]:
def evaluar(idioma, k=3):
    consultas = pd.read_csv(cargar_datos(f"faq_consultas_{idioma}.csv"))
    filas = []
    for _, fila in consultas.iterrows():
        relevantes = {int(x) for x in str(fila["id_documento_relevante"]).split(";")}
        q_tok = tokenizar(fila["consulta"], idioma)
        orden_c, _ = buscar_coseno(q_tok, X_C[idioma], IDX_C[idioma], IDF_TFIDF[idioma], k=len(FAQ[idioma]))
        orden_b, _ = buscar_bm25(q_tok, BM25_C[idioma], IDX_C[idioma], k=len(FAQ[idioma]))
        ids_c = [int(FAQ[idioma].iloc[i]["id"]) for i in orden_c]
        ids_b = [int(FAQ[idioma].iloc[i]["id"]) for i in orden_b]
        filas.append(dict(tipo=fila["tipo"], p_c=precision_en_k(ids_c, relevantes, k), mrr_c=rango_reciproco(ids_c, relevantes),
                           p_b=precision_en_k(ids_b, relevantes, k), mrr_b=rango_reciproco(ids_b, relevantes)))
    return pd.DataFrame(filas)

RESULTADOS = {idioma: evaluar(idioma) for idioma in IDIOMAS}
etq_p3 = "precisión@3"
etq_todas = "todas"
for idioma, res in RESULTADOS.items():
    print("=" * 8, idioma)
    for tipo, sub in [(etq_todas, res), ("facil", res[res.tipo == "facil"]), ("dificil", res[res.tipo == "dificil"])]:
        print(f"  {tipo:>8} (n={len(sub):2d}) | TF-IDF: {etq_p3}={sub.p_c.mean():.3f} MRR={sub.mrr_c.mean():.3f}"
              f" | BM25: {etq_p3}={sub.p_b.mean():.3f} MRR={sub.mrr_b.mean():.3f}")
    assert res[res.tipo == "facil"].mrr_c.mean() > res[res.tipo == "dificil"].mrr_c.mean()   # las fáciles deben ir mejor que las difíciles

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(10, 4))
for eje, idioma in zip(ejes, IDIOMAS):
    res = RESULTADOS[idioma]
    medias = res.groupby("tipo")[["mrr_c", "mrr_b"]].mean()
    medias.columns = ["TF-IDF", "BM25"]
    medias.plot.bar(ax=eje, rot=0, ylim=(0, 1.05), legend=(idioma == IDIOMAS[0]))
    titulo_mrr = "MRR por tipo de consulta"
    eje.set_title(f"{titulo_mrr} ({idioma})")
    eje.set_ylabel("MRR")
plt.tight_layout(); plt.show()

### ¿Qué observas?

En las consultas fáciles, el MRR es 1,0 en los dos idiomas y con los dos buscadores: basta con compartir vocabulario. En las difíciles cae a la mitad o menos. La precisión@3 se queda baja incluso en las fáciles porque casi todas tienen un solo documento relevante (el máximo posible con `k=3` es 1/3). TF-IDF y BM25 dan resultados muy parecidos en esta colección de documentos cortos: BM25 no es automáticamente mejor.

---
## 5. Un fallo concreto: sinónimos y paráfrasis

Miramos de cerca dos consultas "difíciles": una que usa un sinónimo de una palabra clave del documento, y otra que parafrasea toda la frase.

In [ ]:
# datos-es
CONSULTAS_DEMO = {"es": ["cómo recupero mi contraseña olvidada", "olvidé mi clave de acceso a la cuenta", "cómo colaboro con mis compañeros de trabajo en un proyecto"],
                  "it": ["come recupero la mia password dimenticata", "ho dimenticato la chiave di accesso al mio account", "come collaboro con i miei colleghi su un progetto"]}

In [ ]:
for idioma in IDIOMAS:
    print("=" * 8, idioma)
    for consulta in CONSULTAS_DEMO[idioma]:
        q_tok = tokenizar(consulta, idioma)
        orden_c, punt_c = buscar_coseno(q_tok, X_C[idioma], IDX_C[idioma], IDF_TFIDF[idioma], k=3)
        print(f"  «{consulta}»")
        for i, p in zip(orden_c, punt_c):
            print(f"      {int(FAQ[idioma].iloc[i]['id']):3d}  {FAQ[idioma].iloc[i]['titulo']:<55} {p:.3f}")

### ¿Qué observas?

En español, *"olvidé mi clave de acceso a la cuenta"* no encuentra el documento sobre recuperar la contraseña entre los 3 primeros: ese documento nunca usa la palabra `clave`. *"cómo colaboro con mis compañeros de trabajo en un proyecto"* tampoco encuentra el documento sobre compartir carpetas con el equipo: ninguna palabra coincide. Es el mismo fenómeno explicado en la sección 9 del capítulo, con las cifras completas de posición y MRR.

---
## Resumen

- La **recuperación de información** ordena documentos por relevancia frente a una consulta; TF-IDF+coseno y BM25 son dos formas de puntuar esa relevancia por **coincidencia léxica**.
- El **IDF probabilístico de BM25** (`log((N-df+0,5)/(df+0,5))`) puede ser **negativo** cuando un término aparece en más de la mitad de los documentos; se corrige recortando a 0 o con `log(1+...)`.
- **Precisión@k** y **MRR** evalúan un buscador frente a una relevancia fijada **de antemano**; precisión@k está limitada por cuántos documentos relevantes hay.
- TF-IDF y BM25 son el "recuperador" típico de un sistema **RAG**, pero **fallan ante sinónimos y paráfrasis**: es donde entran los embeddings densos (clase 10).

**Tarea:** el notebook de práctica (`clase11_practica_busqueda_recuperacion_clasica.ipynb`), con extracción de palabras clave y resumen extractivo. **Siguiente:** clase 12, proyecto integrador y puente hacia los transformers.